 # Read the file 'audiol.wav'

In [29]:
import librosa
import numpy as np

# Read audio file
x, Fs = librosa.load('../audio1.wav', sr=None)  # x: input signal, Fs: sampling rate (Hz)
L_x = len(x)  # Length of input signal
t = np.arange(L_x) / Fs  # Time vector for plotting (seconds)


# Initialize output signal y[n]
# Explanation for the Code in Cell 2

This cell calculates the delays in samples and generates an output signal `y[n]` with echoes. Here's a breakdown of the code:

1. **Calculate Delays in Samples**:
    - `d1`, `d2`, and `d3` represent delays of 1 second, 2 seconds, and 3 seconds, respectively, in terms of samples. These are calculated using the sampling rate `Fs`.

2. **Determine Maximum Delay**:
    - The variable `max_delay` is set to the maximum of `d1`, `d2`, and `d3`. This ensures that the output signal `y[n]` is long enough to accommodate all echoes.

3. **Initialize Output Signal**:
    - The output signal `y[n]` is initialized as a zero array with a length of `L_x + max_delay`, where `L_x` is the length of the input signal `x`.

4. **Generate Echoes**:
    - The original signal `x` is added to `y[n]` starting at index 0.
    - The first echo, scaled by 0.9, is added to `y[n]` starting at index `d1`.
    - The second echo, scaled by 0.8, is added to `y[n]` starting at index `d2`.
    - The third echo, scaled by 0.7, is added to `y[n]` starting at index `d3`.

This process creates an output signal `y[n]` that contains the original signal along with three delayed echoes.

In [30]:
# Calculate delays in samples
d1 = int(np.round(Fs))  # 1-second delay
d2 = int(np.round(2 * Fs))  # 2-second delay
d3 = int(np.round(3 * Fs))  # 3-second delay

y = np.zeros(L_x + max_delay)  # Ensure y[n] is long enough

# Generate y[n]
y[:L_x] += x  # Original signal
y[d1:d1+L_x] += 0.9 * x  # First echo
y[d2:d2+L_x] += 0.8 * x  # Second echo
y[d3:d3+L_x] += 0.7 * x  # Third echo

This cell defines the impulse response `h[n]` and plots it. Here's a breakdown of the code:

1. **Impulse Response Initialization**:
    - The length of the impulse response `L_h` is set to `max_delay + 1` to ensure it can accommodate all delays.
    - The impulse response `h[n]` is initialized as a zero array of length `L_h`.

2. **Define Impulse Response**:
    - The value at `n=0` is set to 1, representing the original signal.
    - The values at `n=d1`, `n=d2`, and `n=d3` are set to 0.9, 0.8, and 0.7, respectively, representing the scaled echoes at 1-second, 2-second, and 3-second delays.

3. **Plot Impulse Response**:
    - The time vector `n` is divided by the sampling rate `Fs` to convert sample indices to time in seconds.
    - The impulse response `h[n]` is plotted using `matplotlib` with appropriate labels, title, and grid for better visualization.
    - The plot is saved as an image file named `impulse_response.png` and then closed to free up resources.

In [31]:
import matplotlib.pyplot as plt

# Impulse response h[n]
L_h = max_delay + 1
h = np.zeros(L_h)

h[d1] = 0.9  # n=d1
h[d2] = 0.8  # n=d2
h[d3] = 0.7  # n=d3

# Plot impulse response
n = np.arange(L_h)
plt.figure()
plt.stem(n / Fs, h)
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.title('Impulse Response h[n]')
plt.grid(True)
plt.savefig('impulse_response.png')
plt.close()

# Explanation for the Code in Cell 4

This cell performs the convolution of the input signal `x` with the impulse response `h` to generate the echoed signal `y_conv`. Here's a breakdown of the code:

1. **Convolution**:
    - The `signal.convolve` function from the `scipy` library is used to compute the convolution of `x` and `h`. The `mode='full'` ensures that the output signal `y_conv` contains all possible overlaps between `x` and `h`.

2. **Save Output**:
    - The echoed signal `y_conv` is saved to a file named `echo.wav` using the `sf.write` function from the `soundfile` library. The sampling rate `Fs` is used to ensure the correct playback speed.

3. **Plot Echoed Signal**:
    - The time vector `t_y` is generated to match the length of `y_conv` for plotting.
    - The echoed signal `y_conv` is plotted against `t_y` using `matplotlib`. The plot is labeled and titled appropriately, and the grid is enabled for better visualization.
    - The plot is saved as an image file named `echoed_signal.png` and then closed to free up resources.

In [32]:
from scipy import signal
import soundfile as sf

# Convolution
y_conv = signal.convolve(x, h, mode='full')


sf.write('echo.wav', y_conv, Fs)

# Plot echoed signal
t_y = np.arange(len(y_conv)) / Fs
plt.figure()
plt.plot(t_y, y_conv)
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.title('Echoed Signal y[n]')
plt.grid(True)
plt.savefig('echoed_signal.png')
plt.close()

This cell performs the deconvolution process to recover the original signal `x` from the echoed signal `y_conv`. Here's a breakdown of the code:

1. **Parameters**:
    - `L_y` is the length of the output signal from the convolution, calculated as `L_x + L_h - 1`.
    - `N` is the DFT (Discrete Fourier Transform) length, set to `L_y`.

2. **Compute FFTs**:
    - The FFT (Fast Fourier Transform) of the echoed signal `y_conv` is computed and stored in `Y`.
    - The FFT of the impulse response `h` is computed and stored in `H`.

3. **Deconvolution**:
    - The deconvolution is performed in the frequency domain by dividing `Y` by `H`. A small value `epsilon` is added to `H` to avoid division by zero.

4. **Inverse FFT**:
    - The inverse FFT of the deconvolved signal `X_est` is computed to obtain the estimated signal `x_est` in the time domain.

5. **Trim to Original Length**:
    - The recovered signal `x_est` is trimmed to the original length `L_x` and only the real part is retained to avoid small imaginary components due to numerical errors.

6. **Save Output**:
    - The recovered signal `x_est` is saved to a file named `output.wav` using the sampling rate `Fs`.

7. **Plot Recovered Signal**:
    - The time vector `t_x` is used to plot the recovered signal `x_est` against time.
    - The plot is labeled and titled appropriately, and the grid is enabled for better visualization.
    - The plot is saved as an image file named `recovered_signal.png` and then closed to free up resources.


In [33]:
# Parameters
L_y = L_x + L_h - 1
N = L_y  # DFT length

# Compute FFTs
Y = np.fft.fft(y_conv, N)
H = np.fft.fft(h, N)

# Deconvolution
epsilon = 1e-6
X_est = Y / (H + epsilon)

# Inverse FFT

# Trim to original length and save
x_est = x_est[:L_x].real  # Take real part to avoid small imaginary components
sf.write('output.wav', x_est, Fs)

# Plot recovered signal
t_x = np.arange(L_x) / Fs
plt.figure()
plt.plot(t_x, x_est)
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.title('Recovered Signal x[n]')
plt.grid(True)
plt.savefig('recovered_signal.png')
plt.close()